# 🎵 Prosodic Agent Training (XGBoost)

Trains the Prosodic XGBoost classifier using pre-extracted tabular features.

## Class Imbalance Strategies
The ASVspoof5 dataset has severe imbalance (~1:4 bonafide:spoof).
This notebook allows you to easily toggle between the two best strategies:
1. **SMOTE**: Synthesizes minority class data to achieve a 1:1 ratio.
2. **Scale Pos Weight**: Adjusts the loss gradient to penalize errors on the minority class heavier.

In [ ]:
!pip install -q xgboost pandas numpy scikit-learn matplotlib scipy joblib imbalanced-learn seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/40_PER_22_Data')
CSV_PATH = DRIVE_DIR / 'prosodic_features.csv'
OUT_DIR = DRIVE_DIR / 'prosodic_results'
OUT_DIR.mkdir(exist_ok=True)

print(f'Loading data from {CSV_PATH}...')
df = pd.read_csv(CSV_PATH)
print(f'Total rows: {len(df):,} | Columns: {df.shape[1]}')
print(f"Original Label distribution:\n{df['label'].value_counts()}")

In [ ]:
from imblearn.over_sampling import SMOTE

# ──────── CONFIGURATION ────────
USE_SMOTE = True
# ─────────────────────────────

# Split train / test
META_COLS = {'label', 'filename', 'split'}
feat_cols = [c for c in df.columns if c not in META_COLS]

train_df = df[df['split'] == 'train'].copy()
test_df  = df[df['split'] == 'test'].copy()

X_train = train_df[feat_cols].values.astype(np.float32)
y_train = train_df['label'].values
X_test  = test_df[feat_cols].values.astype(np.float32)
y_test  = test_df['label'].values

# Clean NaN / Inf
np.nan_to_num(X_train, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
np.nan_to_num(X_test,  copy=False, nan=0.0, posinf=0.0, neginf=0.0)

n_spoof    = int((y_train == 1).sum())
n_bonafide = int((y_train == 0).sum())
spw        = n_bonafide / max(n_spoof, 1) if not USE_SMOTE else 1.0

print(f"\n─── Data Pipeline ───")
print(f"Initial Train -> Spoof: {n_spoof:,} | Bonafide: {n_bonafide:,}")

if USE_SMOTE:
    print("Applying SMOTE...")
    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    print(f"After SMOTE   -> Spoof: {(y_train == 1).sum():,} | Bonafide: {(y_train == 0).sum():,}")
else:
    print(f"Using scale_pos_weight = {spw:.4f}")

print(f"Feature columns: {len(feat_cols)}")

In [ ]:
import xgboost as xgb
import json

# Detect GPU
import subprocess, sys
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    has_gpu = result.returncode == 0
except:
    has_gpu = False

tree_method = 'gpu_hist' if has_gpu else 'hist'
device      = 'cuda'     if has_gpu else 'cpu'
print(f'Using tree_method={tree_method}')

model = xgb.XGBClassifier(
    n_estimators          = 1000,          # Increased max trees
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    scale_pos_weight      = spw,           # Will be 1.0 if SMOTE is enabled
    eval_metric           = ['logloss', 'aucpr'], # Track multiple metrics
    early_stopping_rounds = 50,            # Stop if aucpr stops improving
    use_label_encoder     = False,
    tree_method           = tree_method,
    device                = device,
    n_jobs                = -1,
)

model.fit(
    X_train, y_train,
    eval_set = [(X_test, y_test)],
    verbose  = 50,
)

# ── Failsafe save immediately after training ──────────────────────────
model.save_model(str(OUT_DIR / 'best_xgb.json'))
with open(OUT_DIR / 'feature_cols.json', 'w') as f:
    json.dump(feat_cols, f)
print('✅ Model saved!')

In [ ]:
from sklearn.metrics import roc_curve, accuracy_score, f1_score, auc as sklearn_auc, classification_report, confusion_matrix, precision_recall_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns

proba = model.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

fpr, tpr, _ = roc_curve(y_test, proba, pos_label=1)
fnr = 1 - tpr
try:
    eer = brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1)
except:
    eer = float(np.mean(np.abs(fnr - fpr)))

auc_val = sklearn_auc(fpr, tpr)
acc     = accuracy_score(y_test, preds)
f1      = f1_score(y_test, preds)

print(f"\n{'='*50}")
print(f"  Prosodic Agent — Final Results")
print(f"{'='*50}")
print(f"  EER      : {eer*100:.2f}%")
print(f"  AUC      : {auc_val:.4f}")
print(f"  Accuracy : {acc*100:.2f}%")
print(f"  F1 Score : {f1:.4f}")
print(f"{'='*50}")
print(classification_report(y_test, preds, target_names=['bonafide', 'spoof']))

# Save results JSON
results = {'eer': float(eer), 'auc': float(auc_val), 'accuracy': float(acc), 'f1': float(f1)}
with open(OUT_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=4)

# ──────── PLOTS ────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. ROC Curve
axes[0, 0].plot(fpr, tpr, color='blue', label=f'AUC={auc_val:.3f}')
axes[0, 0].plot([0,1],[0,1],'--', color='gray')
axes[0, 0].set_xlabel('False Positive Rate')
axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].set_title('ROC Curve')
axes[0, 0].legend()

# 2. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, proba)
pr_auc = sklearn_auc(recall, precision)
axes[0, 1].plot(recall, precision, color='purple', label=f'PR-AUC={pr_auc:.3f}')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title('Precision-Recall Curve (Good for Imbalance)')
axes[0, 1].legend()

# 3. Confusion Matrix
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0], xticklabels=['Bonafide', 'Spoof'], yticklabels=['Bonafide', 'Spoof'])
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_title('Confusion Matrix')

# 4. Feature Importance
xgb.plot_importance(model, max_num_features=15, ax=axes[1, 1], importance_type='weight', title='Top 15 Feature Importances')

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'prosodic_plots.png'), dpi=150)
plt.show()
print('✅ Results and plots saved!')